In [1]:
import os

BASE_DIR        = r'C:\Users\rahul\OneDrive\Desktop\Projects\Eye Disease Detection\Eye Disease Detection\Eye-Disease-Detection'
DATASET_DIR     = os.path.join(BASE_DIR, 'dataset', 'OCT2017')
TRAIN_DIR       = os.path.join(DATASET_DIR, 'train')
VAL_DIR         = os.path.join(DATASET_DIR, 'val')
TEST_DIR        = os.path.join(DATASET_DIR, 'test')
FEATURES_DIR    = os.path.join(BASE_DIR, 'features')
CHECKPOINTS_DIR = os.path.join(BASE_DIR, 'checkpoints')
RESULTS_DIR     = os.path.join(BASE_DIR, 'results')
CLASSES         = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

print(" Paths configured")

 Paths configured


In [2]:
import torch
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import glob
from tqdm import tqdm

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Using device: {device}")

if device.type == 'cuda':
    print(f" GPU Name: {torch.cuda.get_device_name(0)}")
    print(f" GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print(" WARNING: No GPU detected!")

 Using device: cpu


In [3]:
import torchvision.models as models
import torch.nn as nn

# Load pretrained ResNet50
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Remove the final classification layer
# We want the 2048-d feature vector, not the 1000-class ImageNet output
resnet50.fc = nn.Identity()

# Freeze all layers — we are NOT training, just extracting features
for param in resnet50.parameters():
    param.requires_grad = False

# Move to GPU and set to evaluation mode
resnet50 = resnet50.to(device)
resnet50.eval()

print(" ResNet50 loaded and ready")
print(f" All layers frozen")
print(f" Final layer replaced with Identity (outputs 2048-d vector)")
print(f" Model moved to {device}")

 ResNet50 loaded and ready
 All layers frozen
 Final layer replaced with Identity (outputs 2048-d vector)
 Model moved to cpu


In [4]:
# Define the transform pipeline
# ResNet50 expects 224x224 images normalized with ImageNet mean and std
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean
        std=[0.229, 0.224, 0.225]    # ImageNet std
    )
])

# Custom Dataset class
class OCTDataset(Dataset):
    def __init__(self, split_dir, classes, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for label_idx, cls in enumerate(classes):
            cls_path = os.path.join(split_dir, cls)
            images = glob.glob(os.path.join(cls_path, '*.jpeg'))
            if not images:
                images = glob.glob(os.path.join(cls_path, '*.jpg'))
            if not images:
                images = glob.glob(os.path.join(cls_path, '*.JPEG'))

            for img_path in images:
                self.image_paths.append(img_path)
                self.labels.append(label_idx)

        print(f"  Loaded {len(self.image_paths)} images from {split_dir.split('/')[-1]}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            img = Image.open(img_path).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except Exception as e:
            # Return a blank image if corrupted
            img = torch.zeros(3, 224, 224)
        return img, label, img_path

# Create datasets for all three splits
print(" Loading datasets...")
train_dataset = OCTDataset(TRAIN_DIR, CLASSES, transform)
val_dataset   = OCTDataset(VAL_DIR,   CLASSES, transform)
test_dataset  = OCTDataset(TEST_DIR,  CLASSES, transform)

print(f"\n Total images ready for feature extraction:")
print(f"   Train : {len(train_dataset)}")
print(f"   Val   : {len(val_dataset)}")
print(f"   Test  : {len(test_dataset)}")

 Loading datasets...
  Loaded 83484 images from C:\Users\rahul\OneDrive\Desktop\Projects\Eye Disease Detection\Eye Disease Detection\Eye-Disease-Detection\dataset\OCT2017\train
  Loaded 32 images from C:\Users\rahul\OneDrive\Desktop\Projects\Eye Disease Detection\Eye Disease Detection\Eye-Disease-Detection\dataset\OCT2017\val
  Loaded 968 images from C:\Users\rahul\OneDrive\Desktop\Projects\Eye Disease Detection\Eye Disease Detection\Eye-Disease-Detection\dataset\OCT2017\test

 Total images ready for feature extraction:
   Train : 83484
   Val   : 32
   Test  : 968


In [5]:
# Define the transform pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Custom Dataset class
class OCTDataset(Dataset):
    def __init__(self, split_dir, classes, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for label_idx, cls in enumerate(classes):
            cls_path = os.path.join(split_dir, cls)

            # Collect all image extensions
            images = []
            for ext in ['*.jpeg', '*.jpg', '*.JPEG', '*.JPG', '*.png', '*.PNG']:
                images.extend(glob.glob(os.path.join(cls_path, ext)))

            # Remove duplicates just in case
            images = list(set(images))

            for img_path in images:
                self.image_paths.append(img_path)
                self.labels.append(label_idx)

        print(f"  Loaded {len(self.image_paths)} images from {split_dir.split('/')[-1]}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            img = Image.open(img_path).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except Exception as e:
            img = torch.zeros(3, 224, 224)
        return img, label, img_path

# Create datasets
print(" Loading datasets...")
train_dataset = OCTDataset(TRAIN_DIR, CLASSES, transform)
val_dataset   = OCTDataset(VAL_DIR,   CLASSES, transform)
test_dataset  = OCTDataset(TEST_DIR,  CLASSES, transform)

print(f"\n Total images ready for feature extraction:")
print(f"   Train : {len(train_dataset)}")
print(f"   Val   : {len(val_dataset)}")
print(f"   Test  : {len(test_dataset)}")

 Loading datasets...
  Loaded 83484 images from C:\Users\rahul\OneDrive\Desktop\Projects\Eye Disease Detection\Eye Disease Detection\Eye-Disease-Detection\dataset\OCT2017\train
  Loaded 32 images from C:\Users\rahul\OneDrive\Desktop\Projects\Eye Disease Detection\Eye Disease Detection\Eye-Disease-Detection\dataset\OCT2017\val
  Loaded 968 images from C:\Users\rahul\OneDrive\Desktop\Projects\Eye Disease Detection\Eye Disease Detection\Eye-Disease-Detection\dataset\OCT2017\test

 Total images ready for feature extraction:
   Train : 83484
   Val   : 32
   Test  : 968


In [6]:
def extract_and_save(dataset, model, batch_size=64, split_name='', save_dir=''):
    # Check if already extracted — if so skip!
    features_path = os.path.join(save_dir, f'{split_name}_features.npy')
    labels_path   = os.path.join(save_dir, f'{split_name}_labels.npy')
    paths_path    = os.path.join(save_dir, f'{split_name}_paths.npy')

    if os.path.exists(features_path) and os.path.exists(labels_path):
        print(f" {split_name} features already exist — loading from Drive...")
        features = np.load(features_path)
        labels   = np.load(labels_path)
        paths    = np.load(paths_path)
        print(f"   Shape: {features.shape}")
        return features, labels, paths

    # Otherwise extract
    dataloader = DataLoader(dataset, batch_size=batch_size,
                           shuffle=False, num_workers=2, pin_memory=True)

    all_features = []
    all_labels   = []
    all_paths    = []

    print(f"\n Extracting features for {split_name}...")
    print(f"   Total images : {len(dataset)}")
    print(f"   Batch size   : {batch_size}")
    print(f"   Total batches: {len(dataloader)}")

    with torch.no_grad():
        for batch_imgs, batch_labels, batch_paths in tqdm(dataloader, desc=split_name):
            batch_imgs = batch_imgs.to(device)
            features   = model(batch_imgs)

            all_features.append(features.cpu().numpy())
            all_labels.extend(batch_labels.numpy())
            all_paths.extend(batch_paths)

    all_features = np.concatenate(all_features, axis=0)
    all_labels   = np.array(all_labels)

    # Save to Drive
    np.save(features_path, all_features)
    np.save(labels_path,   all_labels)
    np.save(paths_path,    np.array(all_paths))

    print(f"   Extracted and saved to Drive!")
    print(f"   Shape: {all_features.shape}")
    return all_features, all_labels, all_paths

print(" Function defined")

 Function defined


In [7]:
train_features, train_labels, train_paths = extract_and_save(
    train_dataset, resnet50, batch_size=64,
    split_name='train', save_dir=FEATURES_DIR)

val_features, val_labels, val_paths = extract_and_save(
    val_dataset, resnet50, batch_size=64,
    split_name='val', save_dir=FEATURES_DIR)

test_features, test_labels, test_paths = extract_and_save(
    test_dataset, resnet50, batch_size=64,
    split_name='test', save_dir=FEATURES_DIR)

print("\n All done!")
print(f"   Train : {train_features.shape}")
print(f"   Val   : {val_features.shape}")
print(f"   Test  : {test_features.shape}")

 train features already exist — loading from Drive...
   Shape: (83484, 2048)
 val features already exist — loading from Drive...
   Shape: (32, 2048)
 test features already exist — loading from Drive...
   Shape: (968, 2048)

 All done!
   Train : (83484, 2048)
   Val   : (32, 2048)
   Test  : (968, 2048)
